# 第 74 课｜Prefix Cache：命中身份、共享授权与失效策略

**上一课：** 我们把真实 K/V 存在物理分页池，核对了增量前向与无缓存 full forward，学会了引用计数和 COW。

**这一课：** 不同请求怎样复用同一段已经算好的 KV？为什么**相同提示文字**不一定意味着可共享？当模型、LoRA、模板、媒体条件或租户边界变化时，如何拒绝旧缓存？

将完成：

```text
真实Chat Template token IDs
          ↓
可信身份：权重+LoRA+模板+分词+计算配置+隔离组+媒体摘要
          ↓
完整块链式SHA-256索引（不能缓存未满块）
          ↓
共享物理KV页+保留末端logits
          ↓
剩余prompt真实前向／继续解码
          ↓
引用计数、LRU、失效、OOM回滚与全部清零
```

默认**完全离线、CPU、FP32、手写小模型与LoRA**。不是生产级vLLM/PagedAttention GPU Kernel，不报告显卡毫秒或吞吐加速。模拟可信共享范围不构成真实HTTP身份认证。

## 0. 使用说明与最低环境

准备已有 PyTorch 环境。Notebook 内嵌所有模型、分页池与缓存代码，**不依赖第73课文件，也不依赖配套 `.py`**，不会自动下载模型。

建议先看「哈希链和隔离」再运行完整演练。可从新内核「全部运行」。默认执行离线演练；若只读机制，改 `RUN_COMPLETE_DEMO=False`，不会伪造任何训练或演练报告。

这节课没有参数训练：两个独立LoRA模型都被冻结，前缀缓存只用于推理。最后有三道默认关闭的独立练习与完整参考答案。

脚本的执行方式：

```bash
python lesson_74_prefix_cache_identity_isolation_invalidation.py --checks-only
python lesson_74_prefix_cache_identity_isolation_invalidation.py --demo-only
python lesson_74_prefix_cache_identity_isolation_invalidation.py
python lesson_74_prefix_cache_identity_isolation_invalidation.py --reload-run lesson_74_outputs/实际目录
```

每次完整运行在输出根目录中创建新的时间戳目录，不覆盖现有权重/审计报告。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第74课：Prefix Cache哈希链、复用身份、授权隔离、失效与淘汰。

独立离线PyTorch教学实验。真实微型因果Transformer/LoRA在物理KV张量里增量读写，
与全上下文前向核对；在真实已计算KV页上实现完整块前缀复用。
没有PagedAttention融合GPU内核、真实GPU性能数据或生产级服务并发。
"""
from __future__ import annotations

# %% 01 环境、确定性身份与范围
import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter, OrderedDict
import secrets
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson74-prefix-cache-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "vllm_prefix_cache": "https://docs.vllm.ai/en/latest/design/prefix_caching/",
    "vllm_security": "https://docs.vllm.ai/en/stable/usage/security/",
    "vllm_internals": "https://vllm.ai/blog/2025-09-05-anatomy-of-vllm",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(7201)
print('Python:',platform.python_version(),'PyTorch:',torch.__version__)
print('CUDA可用:',torch.cuda.is_available())
print('默认只读CPU，不执行GPU性能测试。')

Python: 3.13.5 PyTorch: 2.10.0+cpu
CUDA可用: False
默认只读CPU，不执行GPU性能测试。


## 1. 首先区分：正文文本、模板边界与最终 token IDs

完整前缀的计算结果由**最终输入token序列**决定，而不是只由用户可见文字决定。

例如这些变化不能粗暴地认为是同一前缀：系统消息改变、角色边界改变、BOS/EOS或 generation prompt 改变、tokenizer规则更新、position或attention实现不同。

本课沿用自建的教学 UTF-8 字节合并分词器，结构 token 使用独立ID，用户文本中出现字面 `<im_start>` 不会得到控制ID。不是任何真实Qwen官方tokenizer。实际生产必须使用部署模型的**官方Chat Template及tokenizer**，并把实际模板与版本身份固定在运行合同里。

下面代码不在字符串中猜标点分割点，而是先由显式 Chat Template 生成 token IDs。

In [3]:
# %% 02 固定教学分词器与真实因果/LoRA模型（独立于前课文件）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
tok = ByteMergeChatTokenizer()
messages=[{'role':'system','content':'Reply briefly.'},
          {'role':'user','content':'Hello world!'}]
ids=tok.apply_chat_template(messages,add_generation_prompt=True)
print('token总数:',len(ids),'前12个ID:',ids[:12])
print('正文里字面控制符编码后是否有真正结构ID:', tok.im_start_token_id in tok.encode('<im_start>'))
print('codec契约摘要长度:',len(digest(tok.contract())))

token总数: 31 前12个ID: [1, 3, 5, 98, 117, 128, 124, 137, 48, 114, 130, 121]
正文里字面控制符编码后是否有真正结构ID: False
codec契约摘要长度: 64


## 2. 模型：真正算过的 KV 才能缓存

本课没有把“同一串token索引”当作可复用 KV 的证明。**先用真实的两层因果 Transformer、Q/V LoRA 算出每层 K/V**，之后才把已经填好的物理页编入 cache index。

两个只读版本 `v0001` 与 `v0002` 同基座但 LoRA 权重不同。它们的同一prompt token序列不能共享一般情况下的 KV：即使某一层的K/V偶然一样，后续层也可能受更新后的隐藏状态影响。

模型使用绝对位置和全因果注意力，全部Dropout为0。没有GQA、RoPE、量化、连续批处理或GPU融合内核。保存的真实模型状态参与缓存身份，不只使用人类可读的版本名。

In [5]:
# %% 03 同基座不同Adapter的只读模型版本
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [6]:
tok,models=build_models(7201)
print('模型版本:',sorted(models))
print('基座一致:',base_hash(models['v0001'])==base_hash(models['v0002']))
print('完整权重不同:',state_hash(models['v0001'].state_dict())!=state_hash(models['v0002'].state_dict()))
print('模型全部冻结:',all(not p.requires_grad for m in models.values() for p in m.parameters()))

模型版本: ['v0001', 'v0002']
基座一致: True
完整权重不同: True
模型全部冻结: True


## 3. 物理KV池：空闲页、会话引用、缓存pin是三个概念

上一课的物理页结构在本课独立复现：

```text
[block, layer, K/V, token_slot, kv_head, head_dim]
```

本课仍是 CPU Tensor 实际预分配，`float32`，`layers=2、heads=4、head_dim=8、block_size=4`。

单token KV = 2 × 2 × 4 × 8 × 4 = **512 字节**；单块四token = **2048 字节**。

**会话引用**表示活动请求仍在用这页；**缓存pin**表示即使请求结束，仍保留一份可复用的完整块。两者都需计入 refcount。

当缓存pin移除但仍有会话引用，不能清零该页；只有最后一个引用消失才能清零回到空闲池。缓存占用的空闲页在有压力时可以被LRU驱逐，活跃会话使用中的页不能被抢走。

本课把缓存对象本身所占的Python metadata排除在物理 KV 张量字节预算之外，报告会专门说明这个边界。

In [7]:
# %% 04 实体KV页池：容量、块表、引用计数与回滚
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

In [8]:
cfg = models['v0001'].cfg
p = PagedKVPool(10,4,cfg,set(models))
print('每tokenKV字节:',p.bytes_per_token)
print('每物理block字节:',p.bytes_per_block)
print('物理Tensor真实预分配字节:',p.data.numel()*p.data.element_size())
print('初始live页:',p.audit()['live_blocks'])

每tokenKV字节: 512
每物理block字节: 2048
物理Tensor真实预分配字节: 20480
初始live页: 0


## 4. 先复现增量计算与全上下文计算一致

`CachedModelRunner.step()` 不是假“KV命中”：它从 token embedding 和绝对位置开始，按层真实计算 Q/K/V，把本token的K/V写入池，读取全部已缓存的过去token，再算attention和最终logits。

为了实现前缀复用，必须额外保存**每个完整块最后一个token的预测logits**。如果新提示恰好全由已缓存的完整块构成，就无需重新计算末尾token；不能只恢复K/V，却丢掉第一步decode的分布。

缓存和全上下文输出会逐请求、在续写之后再次比较，容差为2e−5。该容差不是跨设备的逐位保证。

In [9]:
# %% 05 增量模型执行器：Q/K/V真实写入、块表聚合与全上下文比较
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [10]:
tmp_pool = PagedKVPool(16,4,cfg,set(models))
tmp_runner = CachedModelRunner(tok,models,tmp_pool)
probe_ids=tok.apply_chat_template([{'role':'user','content':'Hello world!'}],
                                  add_generation_prompt=True)
tmp_runner.open_prefill('probe','v0001',probe_ids)
print('增量/full logits最大误差:',tmp_runner.assert_consistent('probe'))
tmp_runner.close('probe')
print('释放后live页:',tmp_pool.audit()['live_blocks'])

增量/full logits最大误差: 3.5762786865234375e-07
释放后live页: 0


## 5. 身份是多维的，不是只有 prompt hash

一段完整块可以复用的最低契约包括：

| 维度 | 何时应拒绝直接复用？ |
|---|---|
| 模型完整权重和LoRA | adapter更新，即使保留相同version字符串 |
| Tokenizer | 词表、合并、特殊token身份变化 |
| Chat Template | 角色结构、generation header、模板修订号变化 |
| 数值计算配置 | attention/position/dtype/层配置不兼容 |
| 最终前缀token IDs | 同一文字被不同方式编码或消息变化 |
| 租户信任域 | 即使文字和模型一样，不被授权共享的请求也不得命中 |
| 额外条件输入 | 如果真实模型还依赖媒体/特征，必须把内容身份带入键 |

这里以 SHA-256 计算权重身份和链式缓存键。理论上任何有限长度摘要都有碰撞可能；教学代码还保留完整前缀token元数据，在读缓存时逐项复核，**不只信任哈希相等**。

vLLM的 Prefix Caching 设计采用父块哈希＋当前完整块tokens＋LoRA/媒体/盐等额外身份，并仅缓存完整块。[vLLM设计文档](https://docs.vllm.ai/en/latest/design/prefix_caching/)

In [11]:
# %% 06 可信共享范围、模型/模板/媒体身份与完整块哈希链
@dataclass(frozen=True)
class CacheRequest:
    """教学内部调用：principal必须由可信网关鉴权后传入，不能让客户端自行声明。"""
    sid: str
    principal: str
    version: str
    messages: tuple[tuple[str,str], ...]
    template_revision: str = "chat-v1"
    media_bytes: bytes | None = None
    add_generation_prompt: bool = True


class TrustedScopeAuthority:
    """服务器控制隔离组与加密随机盐，不接受请求自行提供salt。

    本地示例没有HTTP/登录鉴权。真实调用者必须由网关先确认principal身份，
    本类不防御攻击者能直接调用内部Python对象的情形。
    """
    def __init__(self, principal_groups: dict[str,str]):
        require(type(principal_groups) is dict and bool(principal_groups), '需要非空可信身份组')
        require(all(type(k) is str and k and type(v) is str and v
                    for k,v in principal_groups.items()),'可信身份/共享组非法')
        self._groups=principal_groups.copy()
        self._secrets={g:secrets.token_bytes(32) for g in set(principal_groups.values())}

    def group(self,principal: str) -> str:
        require(type(principal) is str and principal in self._groups,
                '未知或未认证主体不能使用Prefix Cache')
        return self._groups[principal]

    def fingerprint(self,principal: str) -> str:
        group=self.group(principal)
        return hashlib.sha256(self._secrets[group]).hexdigest()

    def rotate(self,group: str) -> str:
        require(group in self._secrets, '未知共享组')
        old=hashlib.sha256(self._secrets[group]).hexdigest()
        self._secrets[group]=secrets.token_bytes(32)
        require(self.fingerprint(next(k for k,v in self._groups.items() if v==group))!=old,
                '加密随机盐轮换未生效')
        return old

    def shared(self,p1:str,p2:str)->bool:
        return self.group(p1)==self.group(p2)


@dataclass(frozen=True)
class CacheIdentity:
    version: str
    model_hash: str
    base_hash: str
    tokenizer_hash: str
    template_hash: str
    scope_hash: str
    media_hash: str
    compute_hash: str


def request_tokens(tok:ByteMergeChatTokenizer,req:CacheRequest)->list[int]:
    require(isinstance(req,CacheRequest),'必须传入CacheRequest')
    require(type(req.messages) is tuple and bool(req.messages),'消息必须为非空tuple')
    require(type(req.add_generation_prompt) is bool,'生成提示选项非法')
    bounded_str(req.template_revision,64,'模板修订号')
    require(bool(req.template_revision),'模板修订号不能为空')
    messages=[]
    for item in req.messages:
        require(type(item) is tuple and len(item)==2 and item[0] in tok.role_ids
                and type(item[1]) is str,'消息必须为(role,content)')
        messages.append({'role':item[0],'content':item[1]})
    return tok.apply_chat_template(messages,add_generation_prompt=req.add_generation_prompt)


def make_identity(req:CacheRequest,authority:TrustedScopeAuthority,tok:ByteMergeChatTokenizer,
                  models:dict[str,TinyChatLM],saved_hashes:dict[str,str]) -> CacheIdentity:
    require(req.version in models and req.version in saved_hashes,'版本没有被可信模型库登记')
    authority.group(req.principal)
    require(type(req.media_bytes) in (bytes,type(None)),'媒体身份必须取自受信预处理的原始字节')
    require(req.media_bytes is None or len(req.media_bytes)<=4096,
            '教学元数据媒体上限4096字节；不允许超大请求键')
    # 禁止在同一个版本名称下原地修改模型而继续读取旧KV。
    model=models[req.version]
    current=state_hash(model.state_dict())
    require(current==saved_hashes[req.version], '热更新必须显式替换并失效缓存，不能原地修改模型')
    return CacheIdentity(
        version=req.version,
        model_hash=current,
        base_hash=base_hash(model),
        tokenizer_hash=digest(tok.contract()),
        template_hash=digest({'family':'toy-message-boundary','revision':req.template_revision,
                              'generation_prompt':req.add_generation_prompt,
                              'role_end':'assistant=eos,others=im_end',
                              'tokenizer':tok.contract()}),
        scope_hash=authority.fingerprint(req.principal),
        media_hash=('none' if req.media_bytes is None else
                    hashlib.sha256(req.media_bytes).hexdigest()),
        compute_hash=digest({'cfg':asdict(model.cfg),'dtype':str(next(model.parameters()).dtype),
                             'position':'absolute','attention':'full causal',
                             'kv_layout':'blocks,layer,kv,slot,heads,dim',
                             'runtime':'toy-cpu-fp32-v1'}),
    )


def prefix_keys(identity:CacheIdentity,ids:list[int],block_size:int)->list[str]:
    require(isinstance(identity,CacheIdentity),'缺少可信缓存身份')
    require(type(block_size) is int and block_size>0,'block_size必须为正整数')
    require(type(ids) is list and all(type(t) is int and t>=0 for t in ids),'token列表非法')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    keys=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        block=ids[start:start+block_size]
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':block})
        keys.append(parent)
    return keys


@dataclass
class CachedBlock:
    key: str
    physical_page: int
    identity: CacheIdentity
    prefix_tokens: tuple[int,...]   # 仅在可信进程内保留，用于哈希碰撞检测；报告禁止展示
    block_tokens: tuple[int,...]
    parent_key: str | None
    end_logits: torch.Tensor
    kv_hash: str
    logits_hash: str
    last_access_tick: int

## 6. 租户隔离不是“让客户端传一个tenant_id”

跨租户 prefix cache 可能泄露**是否已有相同敏感前缀**，即使无法直接读取缓存内容，也可能出现命中时间侧信道。[vLLM安全指南](https://docs.vllm.ai/en/stable/usage/security/)

本课创建一个**可信内部主体→共享组**映射：

```text
alice, alice_worker  → 同一私有范围
bob                  → 另一私有范围
org1, org2           → 明确授权的组织共享范围
```

每个共享范围由服务器端生成32字节加密随机盐。盐**不由请求任意指定，也不打印到报告**；只有受信边界给出的同组主体才会复用。新请求若来自不同租户，即使模型和文字完全相同也会miss。

真实HTTP服务必须先由认证网关验证principal，再把身份传到内部缓存层。本课只有模拟的可信调用接口，**没有实现网络认证、跨进程授权、防时序侧信道的完整威胁模型或密钥持久化**。切勿把这份脚本直接作为生产多租户安全方案。

随机、不可预测的salt有意与固定模型seed分离，所以不同进程的key字节可能不同；我们比较的是授权行为与真实输出，而不是将私有盐写进可复现文件。

In [12]:
authority = TrustedScopeAuthority({'alice':'private_A','alice_worker':'private_A','bob':'private_B'})
print('受信组内部共享:',authority.shared('alice','alice_worker'))
print('不同组禁止共享:',not authority.shared('alice','bob'))
print('server salt字节长度:',sorted(len(s) for s in authority._secrets.values()))
# 不输出原始salt、其摘要或任何用户prompt hash。
try:
    authority.fingerprint('unauthenticated_user')
except LessonError as error:
    print('未认证主体被拒绝:',type(error).__name__)

受信组内部共享: True
不同组禁止共享: True
server salt字节长度: [32, 32]
未认证主体被拒绝: LessonError


## 7. 完整块链：每个键必须继承之前的所有内容

设 block size=4，token序列是11个ID：

```text
[1 2 3 4] [5 6 7 8] [9 10 11]
    B0        B1       未满块
```

本课**只得到两个可复用的块键**：

```text
root = SHA256(可信身份: weights/template/tokenizer/scope/...)
K0   = SHA256(root, 位置0, [1,2,3,4])
K1   = SHA256(K0,   位置4, [5,6,7,8])
```

最后3个token必须在下一请求中重新计算，不能把它们假装成完整缓存块。后来追加一个新token凑满4个，才出现 K2。

改动块0的一个token，会改变 K0及后续所有块；改动某个块内的token，不改变它之前的链。改变LoRA或信任域，则连块0的键也不同。

工业系统可能用更高效的哈希与元数据；这份教学代码偏重可审计性，会保留完整前缀用于碰撞检测，并在读入时校验物理K/V和末端logits的摘要，不能当作高吞吐哈希实现。

In [13]:
req = CacheRequest('demo','alice','v0001',
    messages=(('system','Reply briefly.'),('user','Hello world!')))
identity=make_identity(req,authority,tok,models,{v:state_hash(m.state_dict()) for v,m in models.items()})
prompt_ids=request_tokens(tok,req)
keys=prefix_keys(identity,prompt_ids,4)
print('token数量:',len(prompt_ids),'完整块数量:',len(keys),'未满的尾token:',len(prompt_ids)%4)
print('所有键都是64位小写十六进制摘要:',all(len(k)==64 for k in keys))
print('本课不在屏幕上输出salt或完整cache key。')

token数量: 31 完整块数量: 7 未满的尾token: 3
所有键都是64位小写十六进制摘要: True
本课不在屏幕上输出salt或完整cache key。


## 8. Prefix Cache 的物理所有权：命中、pin、LRU 与回收

与第73课不同，本节对 `PagedKVPool` 增加了持久完整块索引。

- 首次prefill，计算真实 K/V；只有整块提交成功后才能缓存。
- 完整块命中时，不复制K/V张量，只对同一物理页增加会话引用。
- 请求结束后，归还会话引用，但缓存pin仍持有物理页。
- 页面不足：先驱逐**没有活跃会话引用**的LRU缓存pin；所有页仍活跃时明确OOM并回滚失败请求。
- 模型/模板/组失效：去掉缓存pin；活动会话自身引用仍保持正常。
- 最后一个引用释放时清零整页，防止空闲页残留上个请求的KV。

需要分清：**缓存条目数、物理live页数、会话逻辑页引用数**，三者一般不相等。空闲页从未释放其预分配的Tensor，所以底层**总分配字节**仍等于全部物理容量。

部分末页不缓存；但对已经从活动会话fork的未满尾页，仍按上一课的COW规则保护父会话。

In [14]:
# %% 07 对真实KV页加缓存所有权：命中、LRU、失效、物理回收
class PrefixKVPool(PagedKVPool):
    """缓存pin与活动会话引用共同拥有物理页。只发布已提交的完整块。

    与工业vLLM不同：CPU同步、预分配Tensor、无真实异步锁、无融合页注意力核。
    精确前缀tokens用于碰撞检测，存储成本未计入KV字节预算。
    """
    def __init__(self,capacity_blocks:int,block_size:int,model_cfg:ToyConfig,versions:set[str],
                 cache_capacity_blocks:int|None=None):
        super().__init__(capacity_blocks,block_size,model_cfg,versions)
        cache_capacity_blocks=capacity_blocks if cache_capacity_blocks is None else cache_capacity_blocks
        require(type(cache_capacity_blocks) is int and 0<=cache_capacity_blocks<=capacity_blocks,
                '缓存块上限不能超过物理容量')
        self.cache_capacity_blocks=cache_capacity_blocks
        self.cache:dict[str,CachedBlock]={}
        self.clock=0
        self.hit_blocks=0;self.hit_tokens=0;self.miss_requests=0
        self.promotions=0;self.evictions=0;self.invalidations=0
        self.skipped_promotions=0;self.zero_hit_requests=0

    def _tick(self):
        self.clock+=1
        return self.clock

    def _page_hash(self,bid:int)->str:
        return state_hash({'physical_KV':self.data[bid]})

    def _validate_entry(self,e:CachedBlock,identity:CacheIdentity,desired_prefix:tuple[int,...],
                        requested_key:str):
        require(e.key==requested_key and e.identity==identity and
                e.prefix_tokens==desired_prefix,'哈希碰撞或缓存身份/前缀不一致，必须拒绝命中')
        require(e.block_tokens==desired_prefix[-self.block_size:]
                and len(e.block_tokens)==self.block_size,'只能命中完整块')
        require(e.kv_hash==self._page_hash(e.physical_page),'缓存KV字节损坏，拒绝复用')
        require(e.logits_hash==state_hash({'end_logits':e.end_logits}),
                '缓存块末端logits字节损坏，拒绝复用')
        require(self.refs[e.physical_page]>=1,'缓存页已经释放，索引悬空')

    def lookup(self,identity:CacheIdentity,ids:list[int])->list[CachedBlock]:
        keys=prefix_keys(identity,ids,self.block_size)
        found=[]
        for i,key in enumerate(keys):
            e=self.cache.get(key)
            if e is None:break
            self._validate_entry(e,identity,tuple(ids[:(i+1)*self.block_size]),key)
            # 顺序访问同时更新LRU；匹配只返回连续前缀，不能跳过中间缺失块。
            e.last_access_tick=self._tick()
            found.append(e)
        return found

    def _evict_one_idle(self)->bool:
        choices=[e for e in self.cache.values() if self.refs[e.physical_page]==1]
        if not choices:return False
        victim=min(choices,key=lambda e:(e.last_access_tick,e.key))
        del self.cache[victim.key]
        self._decref(victim.physical_page)
        self.evictions+=1
        return True

    def _alloc(self):
        if not self.free:
            self._evict_one_idle()
        return super()._alloc()

    def begin(self,sid,token_id,max_length):
        # 父类在发现free为空时会先报OOM；尝试驱逐idle cache才进入它的begin。
        if sid in self.sessions:
            s=self.sessions[sid]
            if s.pending is None and not self.free:
                slot=len(s.tokens)%self.block_size
                needs_page=slot==0 or (slot>0 and self.refs[s.block_table[-1]]>1)
                if needs_page:self._evict_one_idle()
        return super().begin(sid,token_id,max_length)

    def publish(self,sid:str,index:int,key:str,identity:CacheIdentity,
                prefix_tokens:tuple[int,...],end_logits:torch.Tensor,
                parent_key:str|None)->bool:
        require(sid in self.sessions,'发布会话不存在')
        s=self.sessions[sid]
        require(s.pending is None and s.version==identity.version,'不允许发布未提交/跨版本KV')
        require(type(index) is int and 0<=index<len(s.block_table),'发布块下标错误')
        require(len(prefix_tokens)==(index+1)*self.block_size and
                tuple(s.tokens[:len(prefix_tokens)])==prefix_tokens,'前缀token身份错误')
        require(end_logits.shape==(self.cfg.vocab_size,) and end_logits.dtype==self.dtype and
                bool(torch.isfinite(end_logits).all()),'末端logits形状/数值错误')
        chain=prefix_keys(identity,list(prefix_tokens),self.block_size)
        require(chain and chain[-1]==key and (chain[-2] if len(chain)>1 else None)==parent_key,
                '发布哈希链与token前缀不一致')
        if key in self.cache:
            self._validate_entry(self.cache[key],identity,prefix_tokens,key)
            return False
        if self.cache_capacity_blocks==0:
            self.skipped_promotions+=1
            return False
        if len(self.cache)>=self.cache_capacity_blocks and not self._evict_one_idle():
            self.skipped_promotions+=1
            return False
        bid=s.block_table[index]
        require(self.refs[bid]>=1,'发布物理页没有会话所有权')
        self.refs[bid]+=1             # 独立cache pin：请求结束后仍可命中
        logit=end_logits.detach().cpu().contiguous().clone()
        self.cache[key]=CachedBlock(key=key,physical_page=bid,identity=identity,
             prefix_tokens=prefix_tokens,block_tokens=prefix_tokens[-self.block_size:],
             parent_key=parent_key,end_logits=logit,kv_hash=self._page_hash(bid),
             logits_hash=state_hash({'end_logits':logit}),last_access_tick=self._tick())
        self.promotions+=1
        self.audit()
        return True

    def invalidate(self,predicate:Callable[[CachedBlock],bool]|None=None)->int:
        """移除pin。即使仍有活动会话，只有它们的引用会保留KV。"""
        selected=[key for key,e in self.cache.items() if predicate is None or predicate(e)]
        for key in selected:
            e=self.cache.pop(key)
            self._decref(e.physical_page)
        self.invalidations+=len(selected)
        self.audit()
        return len(selected)

    def advance_clock_and_expire(self,ticks:int,max_idle_ticks:int)->int:
        require(type(ticks) is int and ticks>=0 and type(max_idle_ticks) is int and max_idle_ticks>=0,
                '逻辑时间参数非法')
        self.clock+=ticks
        return self.invalidate(lambda e: self.clock-e.last_access_tick>max_idle_ticks)

    def cache_usage(self)->dict:
        total=self.usage()
        idle_cached=sum(self.refs[e.physical_page]==1 for e in self.cache.values())
        return {'cache_blocks':len(self.cache),'cache_capacity_blocks':self.cache_capacity_blocks,
                'idle_cached_blocks':idle_cached,'active_cached_blocks':len(self.cache)-idle_cached,
                'cache_pins':len(self.cache),'cache_held_bytes':len(self.cache)*self.bytes_per_block,
                'hit_blocks':self.hit_blocks,'hit_tokens':self.hit_tokens,
                'promotions':self.promotions,'evictions':self.evictions,
                'invalidations':self.invalidations,'skipped_promotions':self.skipped_promotions,
                'live_blocks':total['live_blocks'],'free_blocks':total['free_blocks'],
                'session_count':len(self.sessions),'physical_storage_bytes':total['physical_storage_bytes']}

    def usage(self)->dict:
        result=super().usage()
        result['cache_pins']=len(self.cache)
        result['cache_held_bytes']=len(self.cache)*self.bytes_per_block
        # 物理页可以只由缓存持有，因此旧课的logical_refs-live会变负：需把pin计入。
        result['saved_physical_blocks_by_sharing']=(result['logical_page_references']+
                                                     len(self.cache)-result['live_blocks'])
        return result

    def audit(self)->dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话模型版本不在白名单')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '会话页数与token长度不匹配')
            require(len(s.block_table)==len(set(s.block_table)),'同一会话重复引用页')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'页ID越界')
                expected[bid]+=1
        cache_pages=set()
        for key,e in self.cache.items():
            require(key==e.key and e.physical_page not in cache_pages,'重复pin或缓存键/页错配')
            cache_pages.add(e.physical_page)
            require(0<=e.physical_page<self.capacity_blocks,'缓存物理页非法')
            require(len(e.prefix_tokens)%self.block_size==0 and len(e.block_tokens)==self.block_size,
                    '缓存只允许完整块')
            expected[e.physical_page]+=1
        require(expected==self.refs,'会话引用+缓存pin不等于实际物理引用数')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free-list不守恒')
        require(len(set(self.free))==len(self.free),'free-list有重复')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),'空闲页未清零')
        u=self.usage()
        require(u['saved_physical_blocks_by_sharing']>=0,'物理共享账本负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '字节账本不守恒')
        return u

## 9. 命中后的真实prefill和末端分布

新请求先逐块查找最长连续命中链，把命中的完整块挂到自己的块表。其余后缀逐token真实计算，再发布新完成的块；若整个prompt都是完整缓存块，读取最后一个块保存的末端logits即可继续。

对所有请求同时执行 `assert_consistent()`：已经复用的增量K/V和同一个模型的**全上下文前向**应给出相同最终logits。

教学接口将命中数作为内部调试统计返回；**生产多租户API不应把其他主体的命中细节、key、盐或原始缓存tokens暴露给普通用户**。

In [15]:
# %% 08 前缀命中后继续真实预填充，并与完整因果前向核对
class PrefixCacheRunner:
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],
                 pool:PrefixKVPool,authority:TrustedScopeAuthority):
        require(isinstance(pool,PrefixKVPool),'需要有缓存所有权的KV页池')
        self.tok=tok;self.models=models;self.pool=pool;self.authority=authority
        self.runner=CachedModelRunner(tok,models,pool)
        self.bound:dict[str,CacheIdentity]={}
        self.events:list[dict]=[]

    def admit(self,req:CacheRequest)->dict:
        require(isinstance(req,CacheRequest),'请求类型非法')
        self.pool._sid(req.sid)
        require(req.sid not in self.pool.sessions,'sid不可重复入场')
        # 注意：principal须来自已鉴权的内部调用上下文，不应接受HTTP客户端自报。
        ids=request_tokens(self.tok,req)
        require(bool(ids) and len(ids)<=self.pool.cfg.max_length,'提示过长或为空')
        ident=make_identity(req,self.authority,self.tok,self.models,self.runner.model_hashes)
        hits=self.pool.lookup(ident,ids)
        hit_count=len(hits)*self.pool.block_size
        keys=prefix_keys(ident,ids,self.pool.block_size)
        self.pool.open(req.sid,req.version)
        self.bound[req.sid]=ident
        end_logits:dict[int,torch.Tensor]={}
        try:
            state=self.pool.sessions[req.sid]
            for i,e in enumerate(hits):
                self.pool.refs[e.physical_page]+=1
                state.block_table.append(e.physical_page)
                state.tokens.extend(e.block_tokens)
                end_logits[i]=e.end_logits.detach().clone()
            if hits:
                self.runner.last_logits[req.sid]=hits[-1].end_logits.detach().clone()
            self.pool.audit()
            for t in ids[hit_count:]:
                out=self.runner.step(req.sid,t)
                n=len(self.pool.sessions[req.sid].tokens)
                if n%self.pool.block_size==0:
                    end_logits[n//self.pool.block_size-1]=out.detach().clone()
            require(self.pool.sessions[req.sid].tokens==ids,'前缀命中恢复后的token序列错误')
            require(req.sid in self.runner.last_logits,'请求没有任何合法预测logits')
            # 先计算整个prompt，再发布完整块；失败入场不残留不完整页索引。
            parent=None
            for i,key in enumerate(keys):
                prefix=tuple(ids[:(i+1)*self.pool.block_size])
                require(i in end_logits,'完整块末端logits没有计算/回填')
                self.pool.publish(req.sid,i,key,ident,prefix,end_logits[i],parent)
                parent=key
            self.pool.hit_blocks+=len(hits);self.pool.hit_tokens+=hit_count
            if not hits:self.pool.zero_hit_requests+=1
            max_delta=self.runner.assert_consistent(req.sid)
            result={'sid':req.sid,'version':req.version,'prompt_tokens':len(ids),
                    'hit_blocks':len(hits),'hit_tokens':hit_count,'new_tokens_computed':len(ids)-hit_count,
                    'max_logits_abs_delta':max_delta,'cache_blocks_after':len(self.pool.cache),
                    'scope':'internal_authenticated_principal'}
            self.events.append(result.copy())
            return result
        except Exception:
            # 失败不留下活动会话/泄漏物理页；不回滚之前已发生的idle LRU驱逐。
            if req.sid in self.pool.sessions:
                self.runner.close(req.sid)
            self.bound.pop(req.sid,None)
            self.pool.audit()
            raise

    def continue_token(self,sid:str,token_id:int)->float:
        require(sid in self.bound and sid in self.pool.sessions,'不存在的会话')
        identity=self.bound[sid]
        require(identity.model_hash==state_hash(self.models[identity.version].state_dict()),
                '活跃会话期间模型权重不可原地热修改')
        self.runner.step(sid,token_id)
        return self.runner.assert_consistent(sid)

    def fork_session(self,parent:str,child:str,principal:str):
        require(parent in self.bound and parent in self.pool.sessions,'父会话不存在')
        require(child not in self.bound and child not in self.pool.sessions,'子会话已经存在')
        identity=self.bound[parent]
        require(self.authority.fingerprint(principal)==identity.scope_hash,
                'fork只允许相同可信缓存共享范围，不能跨租户')
        self.runner.fork(parent,child,identity.version)
        self.bound[child]=identity
        self.pool.audit()

    def close(self,sid:str):
        require(sid in self.bound,'未授权/已经关闭的会话')
        self.runner.close(sid)
        self.bound.pop(sid)
        self.pool.audit()

    def invalidate_version(self,version:str)->int:
        require(version in self.models,'未知版本')
        return self.pool.invalidate(lambda e:e.identity.version==version)

    def rotate_group(self,group:str)->int:
        old_fp=self.authority.rotate(group)
        return self.pool.invalidate(lambda e:e.identity.scope_hash==old_fp)

    def replace_version(self,version:str,new_model:TinyChatLM)->int:
        require(version in self.models,'未知版本不能替换')
        require(all(s.version!=version for s in self.pool.sessions.values()),
                '有该版本的活跃会话，禁止原地替换/撕裂读写')
        require(isinstance(new_model,TinyChatLM) and new_model.cfg==self.pool.cfg,
                '新模型配置或类型错误')
        require(next(new_model.parameters()).device.type=='cpu'
                and next(new_model.parameters()).dtype==torch.float32,
                '新模型必须是CPU FP32')
        require(all(not p.requires_grad for p in new_model.parameters()),'新模型必须全部冻结')
        replacement=copy.deepcopy(new_model).eval()
        invalidated=self.invalidate_version(version)
        self.models[version]=replacement
        self.runner.models[version]=replacement
        self.runner.model_hashes[version]=state_hash(replacement.state_dict())
        return invalidated

    def final_flush(self)->dict:
        require(not self.pool.sessions and not self.bound,'flush前必须结束全部会话')
        self.pool.invalidate()
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and not bool(self.pool.data.count_nonzero()),
                '全部缓存清空后仍有KV残留')
        return usage

## 10. 一次完整的cold / warm 前缀复用

下面的代码实际运行两个不同的会话：第一个冷prefill；关闭后第二个请求重用物理页。由于最后不足一块，仍需计算少量后缀。

不要只测输出字符串是否一样：我们比较了真实tokenwise KV增量结果和full forward的全部词表logits。

In [16]:
# %% 09 默认演练：命中率、完整块边界、隔离组与模板/媒体身份

def make_demo_env(capacity=90,block_size=4,cache_limit=64,seed=7201):
    tok,models=build_models(seed)
    trusted={'alice':'private_alice','alice_worker':'private_alice',
             'bob':'private_bob','org1':'trusted_org','org2':'trusted_org'}
    authority=TrustedScopeAuthority(trusted)
    pool=PrefixKVPool(capacity,block_size,next(iter(models.values())).cfg,
                      set(models),cache_capacity_blocks=cache_limit)
    service=PrefixCacheRunner(tok,models,pool,authority)
    return tok,models,authority,pool,service


def demo_req(sid:str,principal='alice',version='v0001',body='Hello world!',
             revision='chat-v1',media_bytes=None):
    return CacheRequest(sid=sid,principal=principal,version=version,
                        messages=(('system','Reply briefly.'),('user',body)),
                        template_revision=revision,media_bytes=media_bytes)


def _safe_result(name:str,admit:dict)->dict:
    """演示报告仅保存场景和统计，绝不打印/输出原始消息、盐、媒体或cache key。"""
    return {'scenario':name,'version':admit['version'],
            'prompt_tokens':admit['prompt_tokens'],'hit_blocks':admit['hit_blocks'],
            'hit_tokens':admit['hit_tokens'],'computed_tokens':admit['new_tokens_computed'],
            'max_logit_error':admit['max_logits_abs_delta']}


@torch.inference_mode()
def prefix_correctness_demo(seed=7201):
    tok,models,authority,pool,service=make_demo_env(seed=seed)
    body='Hello world! ' + 'x'*21
    rows=[]
    def go(name,req,extra=()):
        result=service.admit(req)
        errors=[result['max_logits_abs_delta']]
        for token in extra:errors.append(service.continue_token(req.sid,tok.byte_offset+ord(token)))
        rows.append({**_safe_result(name,result),'new_tokens':len(extra),
                     'max_logit_error':max(errors)})
        service.close(req.sid)
        return result
    cold=go('cold_private_alice',demo_req('p0',body=body),'ab')
    warm=go('warm_same_principal',demo_req('p1',body=body),'ab')
    same_scope=go('warm_authorized_worker',demo_req('p2',principal='alice_worker',body=body),'ab')
    changed=go('partial_suffix_change',demo_req('p3',body=body+'!'),'x')
    # 这里的first hit是完整块共前缀；不足一页的相同末尾只能重新计算。
    old_ids=request_tokens(tok,demo_req('unused',body=body))
    new_ids=request_tokens(tok,demo_req('unused2',body=body+'!'))
    common=next((i for i,(a,b) in enumerate(zip(old_ids,new_ids)) if a!=b),min(len(old_ids),len(new_ids)))
    expected=(common//pool.block_size)*pool.block_size
    require(changed['hit_tokens']==expected,'部分末页被错误命中或漏命中')
    isolated=go('different_tenant_miss',demo_req('p4',principal='bob',body=body))
    other_model=go('other_lora_miss',demo_req('p5',version='v0002',body=body))
    other_template=go('template_revision_miss',demo_req('p6',revision='chat-v2',body=body))
    org_cold=go('org_first',demo_req('p7',principal='org1',body=body))
    org_allowed=go('org_explicit_share',demo_req('p8',principal='org2',body=body))
    first_media=go('media_identity_A',demo_req('p9',body=body,media_bytes=b'content A'))
    second_media=go('media_identity_B',demo_req('p10',body=body,media_bytes=b'content B'))
    media_hit=go('media_identity_A_again',demo_req('p11',body=body,media_bytes=b'content A'))
    require(cold['hit_tokens']==0,'首次冷启动不应命中')
    require(warm['hit_tokens']==same_scope['hit_tokens']==len(old_ids)//pool.block_size*pool.block_size,
            '受信共享范围应命中全部完整前缀')
    require(isolated['hit_tokens']==other_model['hit_tokens']==other_template['hit_tokens']==0,
            '租户、LoRA或模板不同不能共享')
    require(org_cold['hit_tokens']==0 and org_allowed['hit_tokens']>0,
            '受信组织共享配置不符')
    require(first_media['hit_tokens']==second_media['hit_tokens']==0 and media_hit['hit_tokens']>0,
            '媒体内容身份未纳入隔离')
    require(all(len(e.block_tokens)==pool.block_size for e in pool.cache.values()),
            '缓存条目不全是完整块')
    before_flush=pool.cache_usage();
    require(before_flush['session_count']==0 and before_flush['cache_blocks']>0,
            '请求结束后应只留下有pin的缓存页')
    final=service.final_flush()
    return {'scenarios':rows,'max_logit_error':max(r['max_logit_error'] for r in rows),
            'full_block_hit_rule_verified':True,'cross_tenant_sharing_refused':True,
            'explicit_org_sharing_verified':True,'different_LoRA_miss':True,
            'template_revision_miss':True,'media_identity_partitioned':True,
            'before_flush':before_flush,
            'after_flush':{'live_blocks':final['live_blocks'],'cache_pins':final['cache_pins'],
                           'nonzero_elements':int(pool.data.count_nonzero())},
            'notes':['媒体bytes只用于身份分区；教学TinyChatLM不是视觉模型',
                     'salt来自可信服务器随机生成，不在报告中出现',
                     '命中token代表避免的tokenwise前向数，不是实测毫秒加速']}


@torch.inference_mode()
def invalidation_demo(seed=7201):
    tok,models,auth,pool,service=make_demo_env(seed=seed)
    body='Hello world! adapter identity.'
    first=service.admit(demo_req('switch_active',body=body))
    cached=len(pool.cache)
    old_live=pool.usage()['live_blocks']
    removed=pool.invalidate(lambda e:e.identity.version=='v0001')
    require(removed==cached>0,'未撤销所有v1缓存pin')
    require(pool.usage()['live_blocks']==old_live,'活动会话不应随缓存失效被清空')
    kept_error=service.runner.assert_consistent('switch_active')
    token_error=service.continue_token('switch_active',tok.byte_offset+ord('y'))
    service.close('switch_active')
    require(pool.usage()['live_blocks']==0,'失效后活动会话结束必须释放KV')
    fresh=service.admit(demo_req('cold_after_invalidate',body=body));service.close('cold_after_invalidate')
    require(fresh['hit_tokens']==0,'显式失效后旧前缀不能命中')
    # 更换可信组的随机salt，旧scope上的缓存被移除，新请求不能通过旧键命中。
    scope_removed=service.rotate_group('private_alice')
    rotated=service.admit(demo_req('after_rotation',body=body));service.close('after_rotation')
    require(scope_removed>0 and rotated['hit_tokens']==0,'轮换盐后仍发生旧域复用')
    # 对同版本别名直接改参数，必须先失效、且不能撕裂活动会话。
    still_active=service.admit(demo_req('active_swap',body=body))
    new_model=copy.deepcopy(models['v0001'])
    with torch.no_grad():
        b=next(p for n,p in new_model.named_parameters() if n.endswith('.B'))
        b[0,0]+=0.3
    new_model.eval().requires_grad_(False)
    rejected=False
    try:service.replace_version('v0001',new_model)
    except LessonError:rejected=True
    require(rejected,'活跃模型版本被错误热替换')
    service.close('active_swap')
    # 保留v2缓存验证：替换v1只清理v1，不能顺手删其他版本。
    v2=service.admit(demo_req('v2_keep',version='v0002',body=body));service.close('v2_keep')
    v2_cached=sum(e.identity.version=='v0002' for e in pool.cache.values())
    old_hash=service.runner.model_hashes['v0001']
    invalidated=service.replace_version('v0001',new_model)
    require(service.runner.model_hashes['v0001']!=old_hash and invalidated>0,
            '显式替换没有更新权重身份和缓存索引')
    require(sum(e.identity.version=='v0002' for e in pool.cache.values())==v2_cached,
            '替换v1误删了v2缓存')
    new_req=service.admit(demo_req('swapped_cold',body=body))
    require(new_req['hit_tokens']==0,'替换同一version label却错误命中了旧权重')
    model_match_error=service.runner.assert_consistent('swapped_cold')
    service.close('swapped_cold')
    v2_warm=service.admit(demo_req('v2_warm',version='v0002',body=body))
    require(v2_warm['hit_tokens']>0,'其他模型版本的缓存应当仍可用')
    service.close('v2_warm')
    # 未显式replace的原地改权重，必须拒绝读取缓存。
    model=service.models['v0001']
    with torch.no_grad():
        p=next(p for n,p in model.named_parameters() if n.endswith('.B'))
        p[0,0]+=0.1
    mutated_rejected=False
    try:service.admit(demo_req('mutated_reject',body=body))
    except LessonError:mutated_rejected=True
    require(mutated_rejected and 'mutated_reject' not in pool.sessions,
            '未经失效原地修改模型权重没有被拒绝')
    final=service.final_flush()
    return {'initial_full_blocks_pinned':cached,'invalidation_under_active_removed':removed,
            'active_session_still_correct_max_diff':max(kept_error,token_error),
            'invalidation_then_miss':fresh['hit_tokens']==0,
            'rotation_removed':scope_removed,'rotation_then_miss':rotated['hit_tokens']==0,
            'hot_swap_while_active_rejected':rejected,'explicit_swap_invalidated':invalidated,
            'swapped_model_first_request_hit_tokens':new_req['hit_tokens'],
            'swapped_model_logit_max_diff':model_match_error,
            'other_version_kept_blocks':v2_cached,'other_version_warm_hit':v2_warm['hit_tokens'],
            'silent_in_place_weight_mutation_rejected':mutated_rejected,
            'after_flush_live_blocks':final['live_blocks']}


@torch.inference_mode()
def eviction_and_failure_demo(seed=7201):
    # 情形A：有限cache容量，完整页发布触发LRU，失效后绝不悬挂脏引用。
    tok,models,auth,pool,service=make_demo_env(capacity=16,block_size=4,cache_limit=3,seed=seed)
    one=service.admit(demo_req('lru_a',body='Hello world!'+'a'*21));service.close('lru_a')
    pinned_start=len(pool.cache)
    two=service.admit(demo_req('lru_b',principal='bob',body='unrelated '+'z'*21));service.close('lru_b')
    require(pool.evictions>=1,'LRU没有驱逐空闲的cache pin')
    lru={'cache_limit':pool.cache_capacity_blocks,'first_promotions':pinned_start,
         'evictions':pool.evictions,'skipped_promotions':pool.skipped_promotions,
         'cache_blocks':len(pool.cache),'last_request_hit_tokens':two['hit_tokens']}
    service.final_flush()
    # 情形B：旧缓存占用的物理页可以按需驱逐，绝不是OOM就清理活跃页。
    tok,models,auth,pool,service=make_demo_env(capacity=9,block_size=4,cache_limit=9,seed=seed)
    first=service.admit(demo_req('pressure1',body='Hello world!'))
    service.close('pressure1')
    before=pool.cache_usage()
    second=service.admit(demo_req('pressure2',principal='bob',body='Hello world!'))
    evict_count=pool.evictions
    require(evict_count>0 and second['hit_tokens']==0,'跨scope物理压力应当驱逐idle缓存而非命中')
    service.close('pressure2')
    pressure={'cached_before':before['cache_blocks'],'evictions_due_to_alloc':evict_count,
              'successful_second_request':True,'cached_after':len(pool.cache)}
    service.final_flush()
    # 情形C：所有页仍被活动请求引用时不得强行驱逐；失败请求要回滚所有分配。
    tok,models,auth,pool,service=make_demo_env(capacity=6,block_size=6,cache_limit=6,seed=seed)
    a=demo_req('pinned_active',body='Hello world!')
    b=demo_req('oom_bob',principal='bob',body='Hello world!')
    ids=request_tokens(tok,a)
    require(len(ids)<=6*6 and len(ids)>5*6,'压测提示长度需要31~36')
    service.admit(a)
    live_before=pool.usage()['live_blocks']
    pinned_keys_before=set(pool.cache)
    failed=False
    try:service.admit(b)
    except NoKVSpace:failed=True
    require(failed and 'oom_bob' not in pool.sessions and pool.usage()['live_blocks']==live_before
            and set(pool.cache)==pinned_keys_before,
            'OOM后未保持活动请求/未错误驱逐被活动引用的缓存pin')
    diff=service.runner.assert_consistent('pinned_active')
    service.close('pinned_active')
    recovered=service.admit(demo_req('bob_after_release',principal='bob',body='Hello world!'))
    require(recovered['hit_tokens']==0,'跨tenant不能借恢复名义共享前缀')
    service.close('bob_after_release')
    oom={'oom_refused':failed,'active_session_survived':True,'active_max_logit_delta':diff,
         'no_zombie_session_after_oom':True,'recovery_without_reset':True,
         'evictions_after_release':pool.evictions}
    service.final_flush()
    # 情形D：空闲TTL按逻辑tick驱逐，不是依赖真实系统时间；零化且没有悬挂索引。
    tok,models,auth,pool,service=make_demo_env(capacity=40,block_size=4,cache_limit=30,seed=seed)
    service.admit(demo_req('ttl_a'));service.close('ttl_a')
    before_ttl=len(pool.cache)
    expired=pool.advance_clock_and_expire(ticks=1000,max_idle_ticks=10)
    again=service.admit(demo_req('ttl_b'));service.close('ttl_b')
    require(expired==before_ttl>0 and again['hit_tokens']==0,'TTL没有移除旧完整块')
    service.final_flush()
    return {'lru':lru,'physical_pressure':pressure,'protected_oom':oom,
            'logical_ttl':{'before_blocks':before_ttl,'expired':expired,
                           'next_hit_tokens':again['hit_tokens']},
            'physical_pool_always_freed_after_explicit_flush':True,
            'note':'tick/页分配是教学事件，不是实际服务延迟或GPU throughput'}

In [17]:
tok,models,authority,pool,service=make_demo_env(capacity=40,block_size=4,cache_limit=25)
cold=service.admit(demo_req('cold_demo'))
service.close('cold_demo')
idle_cache=pool.cache_usage()
warm=service.admit(demo_req('warm_demo'))
print('cold:',{k:cold[k] for k in ('prompt_tokens','hit_tokens','new_tokens_computed','max_logits_abs_delta')})
print('warm:',{k:warm[k] for k in ('prompt_tokens','hit_tokens','new_tokens_computed','max_logits_abs_delta')})
print('中间时刻cache块数:',idle_cache['cache_blocks'],'会话数:',idle_cache['session_count'])
print('两次请求后最大logits误差:',max(cold['max_logits_abs_delta'],warm['max_logits_abs_delta']))
service.close('warm_demo')
final=service.final_flush()
print('显式清空后live页、非零KV元素:',final['live_blocks'],int(pool.data.count_nonzero()))

cold: {'prompt_tokens': 31, 'hit_tokens': 0, 'new_tokens_computed': 31, 'max_logits_abs_delta': 3.5762786865234375e-07}
warm: {'prompt_tokens': 31, 'hit_tokens': 28, 'new_tokens_computed': 3, 'max_logits_abs_delta': 3.5762786865234375e-07}
中间时刻cache块数: 7 会话数: 0
两次请求后最大logits误差: 3.5762786865234375e-07
显式清空后live页、非零KV元素: 0 0


## 11. 更严格的身份对照：什么能命中，什么必须 miss？

完整演练会覆盖：

- **同一主体**重新请求相同prompt：完整前缀命中。
- **同一可信授权组**的另一内部worker：可命中。
- **不同租户**：同文字也不复用。
- **不同LoRA adapter**：同基座也不复用。
- **模板revision不同但token IDs恰好相同**：仍保守拒绝共享。
- **前缀相同但某个后缀字符变化**：仅共享它之前的完整块。
- **媒体内容摘要不同**：索引隔离；模型本身不是多模态模型。

媒体示例只演示**身份键设计**，没有图像编码器、图像embedding或真实多模态前向。未来迁移到真实多模态时，必须以可信预处理内容的身份替代可冒用的客户端UUID，不能只依赖相同的placeholder token。

In [18]:
correctness = prefix_correctness_demo()
print('scenario                         hit tokens  computed tokens  max logits error')
for row in correctness['scenarios']:
    print(f"{row['scenario']:32s} {row['hit_tokens']:6d}        {row['computed_tokens']:6d}       {row['max_logit_error']:.3g}")
print('全部强制释放后live页:',correctness['after_flush']['live_blocks'])

scenario                         hit tokens  computed tokens  max logits error
cold_private_alice                    0            53       3.58e-07
warm_same_principal                  52             1       3.58e-07
warm_authorized_worker               52             1       3.58e-07
partial_suffix_change                48             6       4.77e-07
different_tenant_miss                 0            53       3.58e-07
other_lora_miss                       0            53       3.58e-07
template_revision_miss                0            53       3.58e-07
org_first                             0            53       3.58e-07
org_explicit_share                   52             1       3.58e-07
media_identity_A                      0            53       3.58e-07
media_identity_B                      0            53       3.58e-07
media_identity_A_again               52             1       3.58e-07
全部强制释放后live页: 0


## 12. 失效时不要误伤正在生成的会话

缓存失效要分清两个操作：

1. **移除“供未来命中”的cache pin**：不能再用该索引进入新的请求。
2. **释放“活动会话”的物理页引用**：只有请求自身结束才发生。

只执行第一步，不应该使当前正在生成的会话读到被清零的KV。

本课以实际模型演练三种情况：

- 缓存pin在活跃会话仍引用时失效，父会话继续生成且logits仍正确。
- 可信共享组salt轮换，旧索引失效；未来请求必定miss。
- 同名模型版本热更新：在有活动会话时拒绝；全部关闭后，显式失效对应版本旧缓存，替换权重并核对新logits；**另一LoRA版本的缓存不应被误删**。

直接原地修改模型参数却不更新身份，必须fail-closed。这里每次入场对微型模型重新计算权重摘要，成本在真实大模型上可能非常高；生产环境应使用受信的、不可变发布工件ID和加载/替换生命周期，而不是每请求哈希数十GB权重。

In [19]:
invalidation_report = invalidation_demo()
for name in ('invalidation_under_active_removed','active_session_still_correct_max_diff',
             'rotation_then_miss','hot_swap_while_active_rejected',
             'explicit_swap_invalidated','other_version_warm_hit',
             'silent_in_place_weight_mutation_rejected','after_flush_live_blocks'):
    print(name,':',invalidation_report[name])

invalidation_under_active_removed : 12
active_session_still_correct_max_diff : 3.5762786865234375e-07
rotation_then_miss : True
hot_swap_while_active_rejected : True
explicit_swap_invalidated : 12
other_version_warm_hit : 48
silent_in_place_weight_mutation_rejected : True
after_flush_live_blocks : 0


## 13. LRU、真实物理压力、OOM、逻辑TTL

LRU的驱逐对象必须是**只剩cache pin**的完整页。若活动会话仍持有该页，即使它的缓存键“很旧”，也不能把它用于其他请求。

我们分别运行四个真实CPU KV实例：

1. 缓存条目上限为3，多个不同完整前缀触发idle LRU驱逐。
2. 物理容量只有9页，新的跨租户请求需要更多页；优先逐出旧的idle缓存并完成推理。
3. 物理容量被活动会话占满且无可驱逐的idle页：新请求明确OOM，回滚自己的页表；旧请求继续生成不受破坏。活动会话结束后恢复接入。
4. TTL使用**手动推进的逻辑tick**；不是依赖墙上时间或真实在线访问频率。

成功的资源审计要求：全部会话已关闭、清空全部pin后，live物理页=0，整个预分配KV Tensor非零元素=0。

**注意：** 如果服务永不显式失效，也没有容量压力，idle pin会继续占用页；这不是资源泄漏，属于缓存保留策略。配额、TTL与全局物理容量需要协同设计。

In [20]:
pressure = eviction_and_failure_demo()
print('LRU:',pressure['lru'])
print('物理压力:',pressure['physical_pressure'])
print('受保护OOM:',pressure['protected_oom'])
print('逻辑TTL:',pressure['logical_ttl'])

LRU: {'cache_limit': 3, 'first_promotions': 3, 'evictions': 3, 'skipped_promotions': 20, 'cache_blocks': 3, 'last_request_hit_tokens': 0}
物理压力: {'cached_before': 7, 'evictions_due_to_alloc': 6, 'successful_second_request': True, 'cached_after': 8}
受保护OOM: {'oom_refused': True, 'active_session_survived': True, 'active_max_logit_delta': 3.5762786865234375e-07, 'no_zombie_session_after_oom': True, 'recovery_without_reset': True, 'evictions_after_release': 5}
逻辑TTL: {'before_blocks': 7, 'expired': 7, 'next_hit_tokens': 0}


## 14. CPU缓存账本与性能结论边界

| 统计 | 能证明什么 | **不能**证明什么 |
|---|---|---|
| `hit_tokens` | 跳过了多少逐token的教学模型prefill | 真实TTFT/吞吐加速倍数 |
| `cache_held_bytes` | 被cache pin持有的物理Tensor页额定字节 | Python索引/metadata总内存 |
| `physical_storage_bytes` | CPU Tensor的真实预分配字节 | GPU峰值显存、CUDA allocator行为 |
| `max_logits_abs_delta` | 小型同版本模型KV输出与full一致 | 任意模型/任意模板均正确 |
| `evictions` | idle pin被驱逐次数 | 生产LRU策略最优 |
| `zeroized_releases` | 空闲物理页执行过清零 | GPU并发多租户安全完全证明 |

本课的页、哈希链、授权控制只在CPU单进程同步运行；没有实际线程竞态，也没有真实HTTP、GPU PagedAttention、跨GPU/worker的共享KV传输、缓存持久化或在线监控。

In [21]:
print('每token真实KV字节：', 2*2*4*8*4)
print('每个4-token物理页：', 2*2*4*8*4*4)
print('演练前缀KV最大差异：',correctness['max_logit_error'])
print('结束后所有物理页清零：',correctness['after_flush']['nonzero_elements']==0)

每token真实KV字节： 512
每个4-token物理页： 2048
演练前缀KV最大差异： 4.76837158203125e-07
结束后所有物理页清零： True


## 15. 这不是完整的生产安全边界

vLLM文档指出：多租户前缀缓存的命中可形成时间侧信道；可用针对可信范围的秘密salt减少不应有的跨租户复用。[vLLM 安全说明](https://docs.vllm.ai/en/stable/usage/security/)

但**salt不能替代认证**。如果客户端能冒用别人的principal、获取salt、指定任意共享组，或另一个缓存层使用可碰撞的客户端媒体UUID，仍会产生风险。

本课的 `CacheRequest.principal` 是**内部已经完成鉴权后的身份**，并没有验证现实中的token、session或API key。一个攻击者若可随意调用这些内部Python方法，当然也可能伪造主体；这正是本课不声称实现生产安全的原因。

还需在实际部署中考虑：缓存命中指标的访问控制、审计日志脱敏、secret轮换、计算后端配置、模型工件不可变身份、流量隔离、DoS输入长度限制与真实线程/进程竞态。这里没有进行漏洞利用或外部服务安全测试。

In [22]:
# 检查演练报告，不应包含输入正文、实际salt、cache key或原始媒体bytes。
public_report=canonical(correctness)
for forbidden in ('Hello world!','content A','content B','Reply briefly.'):
    print('报告不暴露用户正文:',forbidden,forbidden not in public_report)
print('报告包含纯统计与示例场景名称，不含实际scope salt。')

报告不暴露用户正文: Hello world! True
报告不暴露用户正文: content A True
报告不暴露用户正文: content B True
报告不暴露用户正文: Reply briefly. True
报告包含纯统计与示例场景名称，不含实际scope salt。


## 16. 自动检查：不仅要测命中率，更要测错误时不串线

下面的固定检查验证：

- token/模板/权重/LoRA/共享范围/媒体身份变化对完整块键的影响；
- 新请求冷启动、命中完整前缀、部分末页、同组共享与跨组miss；
- 实体KV页引用、尾页fork和COW、缓存/活动所有权分离；
- 人为篡改已缓存KV、末端logits和前缀元数据后必须拒绝命中；
- 活跃失效、salt轮换、显式模型热替换与静默模型突变拒绝；
- LRU、TTL、真实物理页不足导致的OOM和回滚；
- 禁用缓存时应仍真实计算，最后显式flush全部页清零。

这些检查是特定教学实现的单元与集成测试，不构成真实多租户安全认证或GPU基准。

In [23]:
# %% 10 固定检查：缓存正确性、来源身份、权限、失效与资源账本
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,value):
        if not bool(value):raise AssertionError(name)
        self.names.append(name)
    def eq(self,name,got,want):self.ok(name,got==want)
    def close(self,name,got,want,atol=2e-5):
        a=torch.as_tensor(got,dtype=torch.float64);b=torch.as_tensor(want,dtype=torch.float64)
        self.ok(name,torch.allclose(a,b,atol=atol,rtol=2e-5))
    def rejects(self,name,callback):
        try:callback()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝而未拒绝')


def run_checks()->dict:
    torch.set_num_threads(1)
    seed_all(7401)
    c=Checks()
    tok,models,authority,pool,service=make_demo_env(capacity=64,block_size=4,cache_limit=46)
    c.eq('two_immutable_weight_versions',set(models),{'v0001','v0002'})
    c.ok('version_weights_different',state_hash(models['v0001'].state_dict())!=state_hash(models['v0002'].state_dict()))
    c.eq('same_base_hash',base_hash(models['v0001']),base_hash(models['v0002']))
    c.ok('all_serve_weights_frozen',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('all_serve_weights_no_grad',all(p.grad is None for m in models.values() for p in m.parameters()))
    c.eq('kv_bytes_per_token',pool.bytes_per_token,512)
    c.eq('kv_bytes_per_block',pool.bytes_per_block,2048)
    c.eq('physical_backing_preallocated',pool.data.numel()*pool.data.element_size(),64*2048)
    c.ok('tokenizer_control_not_user_literal',tok.im_start_token_id not in tok.encode('<im_start>'))
    c.eq('utf8_roundtrip',tok.decode(tok.encode('你好！')),'你好！')
    c.rejects('broken_utf8_cannot_decode',lambda:tok.decode([tok.byte_offset+0xe4]))
    c.ok('tenant_secret_32_bytes',all(len(secret)==32 for secret in authority._secrets.values()))
    c.ok('alice_and_worker_authorized_shared',authority.shared('alice','alice_worker'))
    c.ok('org_pair_authorized_shared',authority.shared('org1','org2'))
    c.ok('different_tenants_private',not authority.shared('alice','bob'))
    c.rejects('unknown_principal_denied',lambda:authority.fingerprint('attacker'))
    c.rejects('arbitrary_external_salt_not_an_API_field',lambda:CacheRequest(
        sid='x',principal='alice',version='v0001',messages=(('user','hi'),),cache_salt='impostor'))
    c.rejects('unbounded_template_revision_rejected',lambda:request_tokens(
        tok,demo_req('x',revision='Z'*100)))
    c.rejects('bad_messages_shape_rejected',lambda:request_tokens(tok,CacheRequest(
        'x','alice','v0001',messages=[('user','hi')])))
    c.rejects('media_bytes_over_limit_rejected',lambda:make_identity(
        demo_req('x',media_bytes=b'x'*4097),authority,tok,models,service.runner.model_hashes))
    c.rejects('client_uuid_not_substitute_for_trusted_media_bytes',lambda:make_identity(
        demo_req('x',media_bytes='some-client-UUID'),authority,tok,models,service.runner.model_hashes))
    c.rejects('negative_cache_limit_rejected',lambda:PrefixKVPool(2,4,models['v0001'].cfg,set(models),-1))
    c.rejects('cache_limit_above_physical_rejected',lambda:PrefixKVPool(2,4,models['v0001'].cfg,set(models),3))
    c.rejects('zero_page_size_rejected',lambda:PrefixKVPool(2,0,models['v0001'].cfg,set(models)))
    req=demo_req('keyA')
    ids=request_tokens(tok,req)
    identity=make_identity(req,authority,tok,models,service.runner.model_hashes)
    k=prefix_keys(identity,ids,4)
    c.eq('only_full_blocks_hashed',len(k),len(ids)//4)
    c.ok('prefix_has_nonempty_chain',len(k)>1)
    c.ok('keys_are_sha256_hex',all(len(x)==64 and all(ch in '0123456789abcdef' for ch in x) for x in k))
    c.eq('trailing_partial_token_does_not_create_new_key',
         prefix_keys(identity,ids[:-1]+[8],4),prefix_keys(identity,ids[:-1],4))
    c.ok('full_new_block_adds_one_key',len(prefix_keys(identity,ids+[8]*(4-len(ids)%4),4))==len(k)+1)
    altered=ids.copy();altered[0]=(altered[0]+1)%tok.vocab_size
    c.ok('change_first_token_changes_whole_chain',all(x!=y for x,y in zip(prefix_keys(identity,altered,4),k)))
    altered=ids.copy();altered[7]=(altered[7]+1)%tok.vocab_size
    c.ok('change_middle_block_keeps_earlier_key',prefix_keys(identity,altered,4)[0]==k[0])
    c.ok('change_middle_block_changes_later_chain',all(x!=y for x,y in zip(prefix_keys(identity,altered,4)[1:],k[1:])))
    c.rejects('zero_size_hash_block_rejected',lambda:prefix_keys(identity,ids,0))
    c.rejects('negative_token_id_rejected',lambda:prefix_keys(identity,ids+[-1],4))
    def ident(principal='alice',version='v0001',revision='chat-v1',media=None):
        return make_identity(demo_req('x',principal=principal,version=version,revision=revision,
                                      media_bytes=media),authority,tok,models,service.runner.model_hashes)
    c.ok('change_tenant_changes_first_key',prefix_keys(ident(principal='bob'),ids,4)[0]!=k[0])
    c.eq('authorized_worker_same_key',prefix_keys(ident(principal='alice_worker'),ids,4),k)
    c.ok('change_lora_changes_first_key',prefix_keys(ident(version='v0002'),ids,4)[0]!=k[0])
    c.ok('change_template_revision_changes_first_key',prefix_keys(ident(revision='chat-v2'),ids,4)[0]!=k[0])
    c.ok('same_tokens_different_media_changes_first_key',prefix_keys(ident(media=b'fake image'),ids,4)[0]!=k[0])
    c.eq('same_media_bytes_same_key',prefix_keys(ident(media=b'fake image'),ids,4),
         prefix_keys(ident(media=b'fake image'),ids,4))
    c.ok('metadata_media_not_model_image_processing',not hasattr(models['v0001'],'vision_tower'))
    c.eq('template_revision_does_not_forge_tokenizer_ids',request_tokens(tok,demo_req('q',revision='chat-v2')),ids)
    c.eq('cache_starts_empty',len(pool.cache),0)
    initial=service.admit(demo_req('cold'))
    c.eq('first_request_zero_hit_tokens',initial['hit_tokens'],0)
    c.eq('cold_computes_full_prompt',initial['new_tokens_computed'],initial['prompt_tokens'])
    c.eq('cold_publish_only_full_blocks',len(pool.cache),initial['prompt_tokens']//4)
    c.eq('first_request_owns_one_private_partial_block',pool.usage()['live_blocks'],math.ceil(len(ids)/4))
    c.eq('full_page_refs_include_cache_pin',pool.refs[pool.sessions['cold'].block_table[0]],2)
    c.ok('cached_full_pages_physically_nonzero',bool(pool.data[pool.sessions['cold'].block_table[0]].count_nonzero()))
    c.ok('full_context_equivalence_first',initial['max_logits_abs_delta']<2e-5)
    c.rejects('reject_duplicate_sid',lambda:service.admit(demo_req('cold')))
    c.rejects('reject_untrusted_user_before_open',lambda:service.admit(demo_req('unknown',principal='mallory')))
    c.ok('untrusted_rejection_no_session','unknown' not in pool.sessions)
    c.rejects('wrong_scope_fork_denied',lambda:service.fork_session('cold','bad_fork','bob'))
    service.fork_session('cold','child','alice_worker')
    c.eq('fork_partial_tail_shared',pool.sessions['cold'].block_table[-1],
         pool.sessions['child'].block_table[-1])
    before_cow=pool.cow_count
    ec=service.continue_token('child',tok.byte_offset+ord('y'))
    c.ok('fork_partial_tail_write_triggers_cow',pool.cow_count>before_cow)
    c.ok('fork_child_logits_consistent',ec<2e-5)
    c.ok('fork_parent_logits_not_corrupted',service.runner.assert_consistent('cold')<2e-5)
    c.ok('fork_parent_and_child_tail_page_different',pool.sessions['cold'].block_table[-1]!=
         pool.sessions['child'].block_table[-1])
    service.close('child');service.close('cold')
    c.ok('after_close_cache_pins_stay_live',pool.usage()['live_blocks']>0)
    c.eq('after_close_no_sessions',len(pool.sessions),0)
    c.eq('after_close_all_pages_idle_cache',pool.usage()['live_blocks'],len(pool.cache))
    warm=service.admit(demo_req('warm'))
    c.eq('same_scope_hit_all_full_prefix',warm['hit_tokens'],len(ids)//4*4)
    c.ok('warm_only_recomputes_partial_tail',warm['new_tokens_computed']<4)
    c.ok('warm_logits_match_full',warm['max_logits_abs_delta']<2e-5)
    c.ok('warm_KV_page_is_same_object',pool.sessions['warm'].block_table[0] in [e.physical_page for e in pool.cache.values()])
    service.close('warm')
    other_tenant=service.admit(demo_req('bob_miss',principal='bob'))
    c.eq('same_text_cross_tenant_does_not_hit',other_tenant['hit_tokens'],0)
    service.close('bob_miss')
    other_adapter=service.admit(demo_req('lora_miss',version='v0002'))
    c.eq('same_text_other_lora_miss',other_adapter['hit_tokens'],0)
    c.ok('different_lora_logits_match_full',other_adapter['max_logits_abs_delta']<2e-5)
    service.close('lora_miss')
    other_template=service.admit(demo_req('tmpl_miss',revision='chat-v2'))
    c.eq('same_tokens_other_template_miss',other_template['hit_tokens'],0)
    service.close('tmpl_miss')
    only_partial=PrefixKVPool(5,64,models['v0001'].cfg,set(models),3)
    rshort=PrefixCacheRunner(tok,models,only_partial,authority)
    short_cold=rshort.admit(demo_req('short0'))
    c.eq('short_less_than_page_never_cache',len(only_partial.cache),0)
    c.eq('short_only_partial_hit_is_zero',short_cold['hit_tokens'],0)
    rshort.close('short0')
    short_warm=rshort.admit(demo_req('short1'))
    c.eq('short_reopen_still_zero_hit',short_warm['hit_tokens'],0)
    rshort.close('short1');rshort.final_flush()
    # 单独KV/末端logits校验：发生损坏必须拒绝，不要返回错误缓存输出。
    cache_key=next(iter(pool.cache));entry=pool.cache[cache_key]
    page=entry.physical_page
    original_scalar=pool.data[page,0,0,0,0,0].clone()
    with torch.no_grad():pool.data[page,0,0,0,0,0]+=1
    c.rejects('tampered_cache_KV_refuses_lookup',lambda:pool.lookup(identity,ids))
    with torch.no_grad():pool.data[page,0,0,0,0,0].copy_(original_scalar)
    c.ok('corrupted_KV_lookup_did_not_create_session','bad_kv' not in pool.sessions)
    oldprefix=entry.prefix_tokens
    entry.prefix_tokens=(999,)*len(oldprefix)
    c.rejects('hash_collision_token_identity_refuses_hit',lambda:service.admit(demo_req('bad_hash')))
    entry.prefix_tokens=oldprefix
    c.ok('bad_hash_no_session_created','bad_hash' not in pool.sessions)
    orig_logits=entry.end_logits.clone()
    entry.end_logits[0]+=1
    c.rejects('tampered_end_logits_refuses_hit',lambda:service.admit(demo_req('bad_logits')))
    entry.end_logits.copy_(orig_logits)
    c.ok('tamper_recovery_without_disabling_checks',service.admit(demo_req('good_after_tamper'))['hit_tokens']>0)
    service.close('good_after_tamper')
    # 提前将被分享的cache pin移除，活动会话仍保持自身引用。
    under=service.admit(demo_req('live_invalidate'))
    pre_live=pool.usage()['live_blocks']
    active_pages=set(pool.sessions['live_invalidate'].block_table)
    invalidated=pool.invalidate(lambda e:e.identity.version=='v0001')
    c.ok('invalidation_removes_cache_entries',invalidated>0)
    c.ok('active_session_not_released_by_invalidation',
         all(pool.refs[bid]>=1 for bid in active_pages) and
         0<pool.usage()['live_blocks']<=pre_live)
    c.ok('active_session_remains_consistent',service.runner.assert_consistent('live_invalidate')<2e-5)
    service.close('live_invalidate')
    c.ok('free_pages_zeroized',all(not bool(pool.data[i].count_nonzero()) for i in pool.free))
    c.rejects('unknown_session_close_rejected',lambda:service.close('unknown'))
    c.rejects('negative_logical_clock_rejected',lambda:pool.advance_clock_and_expire(-1,2))
    c.rejects('negative_ttl_rejected',lambda:pool.advance_clock_and_expire(1,-1))
    c.ok('audit_accounts_for_cache_pins',pool.audit()['cache_pins']==len(pool.cache))
    final=service.final_flush()
    c.eq('final_all_physical_pages_freed',final['live_blocks'],0)
    c.eq('final_cached_pins_freed',final['cache_pins'],0)
    c.eq('final_KV_memory_zeroized',int(pool.data.count_nonzero()),0)
    demo=prefix_correctness_demo()
    c.ok('full_demo_reuses_physical_KV',demo['scenarios'][1]['hit_tokens']>0)
    c.ok('full_demo_explicit_org_share',demo['explicit_org_sharing_verified'])
    c.ok('full_demo_different_template_misses',demo['template_revision_miss'])
    c.ok('full_demo_media_separated',demo['media_identity_partitioned'])
    c.ok('full_demo_KV_full_logits_equivalence',demo['max_logit_error']<2e-5)
    c.eq('full_demo_no_final_leak',demo['after_flush']['live_blocks'],0)
    inv=invalidation_demo()
    c.ok('active_invalidation_keeps_KV',inv['active_session_still_correct_max_diff']<2e-5)
    c.ok('scoped_salt_rotation_misses',inv['rotation_then_miss'])
    c.ok('hot_swap_rejected_when_active',inv['hot_swap_while_active_rejected'])
    c.ok('hot_swap_with_invalidation_works',inv['explicit_swap_invalidated']>0)
    c.ok('other_version_cache_survives_swap',inv['other_version_warm_hit']>0)
    c.ok('silent_model_mutation_detected',inv['silent_in_place_weight_mutation_rejected'])
    eviction=eviction_and_failure_demo()
    c.ok('lru_evicts_idle_cache',eviction['lru']['evictions']>0)
    c.ok('physical_pressure_evicts_idle_cache',eviction['physical_pressure']['evictions_due_to_alloc']>0)
    c.ok('active_cache_cannot_be_stolen_on_OOM',eviction['protected_oom']['oom_refused'])
    c.ok('OOM_rollback_no_zombie_session',eviction['protected_oom']['no_zombie_session_after_oom'])
    c.ok('OOM_then_recovery',eviction['protected_oom']['recovery_without_reset'])
    c.ok('logical_TTL_invalidates',eviction['logical_ttl']['expired']>0)
    c.ok('all_demo_pools_fully_released',eviction['physical_pool_always_freed_after_explicit_flush'])
    # 一个请求恰好全部是完整块：命中后不需要执行任何额外token前向。
    tok,models,auth,pool,svc=make_demo_env(capacity=45,block_size=4,cache_limit=30)
    chosen=None
    for n in range(24):
        r=demo_req('x',body='A'*n)
        if len(request_tokens(tok,r))%4==0:
            chosen=r.messages[1][1];break
    c.ok('find_prompt_exact_full_block',chosen is not None)
    full_cold=svc.admit(demo_req('all_cold',body=chosen));svc.close('all_cold')
    full_hit=svc.admit(demo_req('all_hit',body=chosen))
    c.eq('all_full_hit_computes_zero_new_tokens',full_hit['new_tokens_computed'],0)
    c.ok('all_full_hit_logits_from_saved_boundary',full_hit['max_logits_abs_delta']<2e-5)
    svc.close('all_hit');svc.final_flush()
    # 当禁用缓存时，仍应执行真实计算，而不是假装有缓存。
    tok,models,auth,pool,svc=make_demo_env(capacity=16,block_size=4,cache_limit=0)
    none=svc.admit(demo_req('no_cache_1'));svc.close('no_cache_1')
    again=svc.admit(demo_req('no_cache_2'));svc.close('no_cache_2')
    c.eq('disabled_cache_never_hits',none['hit_tokens']+again['hit_tokens'],0)
    c.eq('disabled_cache_no_promotions',pool.promotions,0)
    c.eq('disabled_cache_no_live_blocks_after_close',pool.usage()['live_blocks'],0)
    svc.final_flush()
    return {'passed':len(c.names),'names':c.names,
            'scope':'CPU真实KV Tensor、可信边界模拟，不是生产安全认证/GPU吞吐'}

In [24]:
checks=run_checks()
print('固定检查通过:',checks['passed'],'项')

固定检查通过: 115 项


## 17. 独立练习（默认开关均为 False）

请**先自己实现**，再运行检查器。三题关注不同的工程契约，不能只对 `hit=True/False` 打印固定结果。

下面先定义三个题目的检查入口；参考答案放到讲义后面，使用不同名字，不会自动替你填练习。

In [25]:
def exercise_a(fn:Callable)->dict:
    c=Checks()
    tok,models,auth,pool,svc=make_demo_env()
    req=demo_req('exercise_a')
    ident=make_identity(req,auth,tok,models,svc.runner.model_hashes)
    ids=request_tokens(tok,req)
    got=fn(ident,ids,4)
    c.eq('A_full_keys',got,prefix_keys(ident,ids,4))
    c.eq('A_ignore_partial',fn(ident,ids[:-1]+[7],4),fn(ident,ids[:-1],4))
    changed=ids.copy();changed[0]+=1
    c.ok('A_chain_depends_on_first_token',fn(ident,changed,4)[-1]!=got[-1])
    changed_scope=make_identity(demo_req('x',principal='bob'),auth,tok,models,svc.runner.model_hashes)
    c.ok('A_chain_depends_on_authorization',fn(changed_scope,ids,4)[0]!=got[0])
    c.rejects('A_invalid_block_size',lambda:fn(ident,ids,0))
    return {'passed':len(c.names),'names':c.names}


def exercise_b(fn:Callable)->dict:
    c=Checks()
    c.eq('B_only_complete_shared_blocks',fn([1,2,3,4,5],[1,2,3,4,7],4),4)
    c.eq('B_no_full_block',fn([1,2,3],[1,2,3],4),0)
    c.eq('B_no_shared_prefix',fn([1,2,3,4],[8,2,3,4],4),0)
    c.eq('B_two_complete_blocks',fn([1,2,3,4,5,6,7,8,9],[1,2,3,4,5,6,7,8],4),8)
    c.rejects('B_invalid_block_size',lambda:fn([1],[1],0))
    return {'passed':len(c.names),'names':c.names}


def exercise_c(fn:Callable)->dict:
    c=Checks()
    c.eq('C_idle_pin_can_be_evicted',fn(0,1),'evictable')
    c.eq('C_in_use_pin_is_protected',fn(2,1),'protected')
    c.eq('C_unpinned_idle_page_is_free',fn(0,0),'free')
    c.eq('C_active_without_cache_pin',fn(1,0),'protected')
    c.rejects('C_refcount_underflow',lambda:fn(-1,0))
    c.rejects('C_pin_without_reference',lambda:fn(0,2))
    return {'passed':len(c.names),'names':c.names}

### 练习A：完整块哈希链

输入 `CacheIdentity`、完整token列表和block_size，输出按顺序的SHA-256键列表。

要求：以全部可信身份构造root，后续每块把**上一个键、当前块位置与恰好完整的一块tokens**纳入哈希；忽略未满尾块。改首token或隔离组必须改变键。block_size非法时必须拒绝。

可阅读 `digest()` 和 `asdict()` 的作用，但先不要复制参考实现。

In [26]:
def prefix_keys_ex(identity, ids, block_size):
    raise NotImplementedError('请完成完整块哈希链；不能把未满尾块发布')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_a(prefix_keys_ex))

### 练习B：最长公共前缀能命中多少token？

只比较两条已经经过同一受信身份授权的 token 列表。先计数从位置0开始连续相等的token数，再向下取整到block_size的整数倍。

注意这里**只计算潜在可复用token数**；真实是否命中，还要查缓存索引、验证物理KV和已有的授权键。

In [27]:
def complete_common_prefix_ex(left, right, block_size):
    raise NotImplementedError('请计算完整块对齐的最长共同token前缀长度')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_b(complete_common_prefix_ex))

### 练习C：物理页是否可以被LRU驱逐？

输入 `active_refs`（活跃会话引用数量）和 `cache_pins`（本课最多为0或1），输出：

- `free`：没有会话也没有缓存pin，本页本来已空闲；
- `evictable`：只有缓存pin，没有会话引用，可以先去pin再归还页；
- `protected`：至少有一个活动会话持有该页，绝不能强制驱逐。

拒绝负引用计数和pin数量大于1。

In [28]:
def physical_page_decision_ex(active_refs, cache_pins):
    raise NotImplementedError('请实现pin与活跃会话引用的驱逐判定')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_c(physical_page_decision_ex))

## 18. 完整参考答案与独立核验

完成自己写的三个函数后，再运行下面的答案。这些参考实现分别独立于工程主函数，不会覆盖练习区。检查器同时包含应拒绝的无效输入，不能只看正常示例。

In [29]:
def answer_a(identity,ids,block_size):
    require(isinstance(identity,CacheIdentity) and type(ids) is list,'身份/token类型不匹配')
    require(type(block_size) is int and block_size>0,'block_size无效')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    result=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':ids[start:start+block_size]})
        result.append(parent)
    return result


def answer_b(left,right,block_size):
    require(type(block_size) is int and block_size>0,'block_size非法')
    common=0
    for a,b in zip(left,right):
        if a!=b:break
        common+=1
    return common//block_size*block_size


def answer_c(active_refs,cache_pins):
    require(type(active_refs) is int and active_refs>=0 and
            type(cache_pins) is int and cache_pins in {0,1},'引用计数非法')
    if cache_pins==0:return 'protected' if active_refs else 'free'
    return 'evictable' if active_refs==0 else 'protected'

In [30]:
print('A:',exercise_a(answer_a))
print('B:',exercise_b(answer_b))
print('C:',exercise_c(answer_c))

A: {'passed': 5, 'names': ['A_full_keys', 'A_ignore_partial', 'A_chain_depends_on_first_token', 'A_chain_depends_on_authorization', 'A_invalid_block_size']}
B: {'passed': 5, 'names': ['B_only_complete_shared_blocks', 'B_no_full_block', 'B_no_shared_prefix', 'B_two_complete_blocks', 'B_invalid_block_size']}
C: {'passed': 6, 'names': ['C_idle_pin_can_be_evicted', 'C_in_use_pin_is_protected', 'C_unpinned_idle_page_is_free', 'C_active_without_cache_pin', 'C_refcount_underflow', 'C_pin_without_reference']}


## 19. 完整演练与可读的报告

默认 `RUN_COMPLETE_DEMO=True`。将两份只读模型的状态保存在 `models.pt`，并保存：

| 文件 | 内容 |
|---|---|
| `correctness.json` | cold/warm、租户/LoRA/模板/媒体隔离和逐请求logits误差 |
| `invalidation.json` | 活跃页失效、盐轮换、受控替换、静默权重突变拒绝 |
| `eviction.json` | LRU、物理压力、OOM回滚与逻辑TTL |
| `weights.json` | 两版本模型完整权重摘要和tokenizer身份 |
| `checks.json` | 运行的固定检查与名称 |
| `reload.json` | 独立干净对象只读重建的对比 |
| `manifest.json` | 环境、范围、完成或失败状态 |

文件**不会保存**原始请求正文、媒体字节、隔离组随机salt或私有cache key；模型权重文件只能从本地可信来源加载。

独立回放会新建模型和空页池、重新生成随机salt，检查逻辑报告与原运行一致；它不是恢复一个仍然在线的KV会话。

In [31]:
def save_models(path:Path,tok,models,seed):
    payload={'version':VERSION,'seed':seed,'cfg':asdict(next(iter(models.values())).cfg),
             'tokenizer':tok.contract(),'hashes':{name:state_hash(m.state_dict()) for name,m in models.items()},
             'states':{name:{k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
                       for name,m in models.items()},'adapter_rank':4,'adapter_alpha':8.0}
    torch.save(payload,path)


def load_models(path:Path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'检查点版本不匹配')
    tok,models=build_models(blob['seed'])
    require(blob['tokenizer']==tok.contract(),'tokenizer契约与权重不匹配')
    require(blob['cfg']==asdict(next(iter(models.values())).cfg),'模型配置不匹配')
    require(blob['adapter_rank']==4 and blob['adapter_alpha']==8.0,'adapter结构不匹配')
    require(set(blob['states'])==set(models)==set(blob['hashes']),'权重版本集合不匹配')
    for name,model in models.items():
        model.load_state_dict(blob['states'][name],strict=True)
        model.requires_grad_(False).eval()
        require(state_hash(model.state_dict())==blob['hashes'][name],'模型权重和摘要不匹配')
    return tok,models


def reload_run(folder:Path):
    folder=Path(folder)
    tok,models=load_models(folder/'models.pt')
    expected=json.loads((folder/'weights.json').read_text(encoding='utf-8'))
    require(expected=={'model_hashes':{v:state_hash(m.state_dict()) for v,m in models.items()},
                       'tokenizer_hash':digest(tok.contract()),
                       'base_hash':base_hash(models['v0001']),'model_seed':expected.get('model_seed')},
            '权重身份报告不一致')
    functions=[('correctness.json',prefix_correctness_demo),('invalidation.json',invalidation_demo),
               ('eviction.json',eviction_and_failure_demo)]
    comparisons={}
    for file,fn in functions:
        # 演练内部使用固定种子重建模型与全新随机salt；仅比业务统计，不比salt/原文。
        actual=fn(seed=expected['model_seed']) if 'model_seed' in expected else fn()
        saved=json.loads((folder/file).read_text(encoding='utf-8'))
        require(actual==saved,'独立进程重载后的演练报告不一致：'+file)
        comparisons[file]={'exact_report_equal':True}
    return {'comparison':comparisons,'weights_validated':True,
            'random_scopes_recreated_by_authority':True,
            'not_live_cache_session_resume':True}


def fixture_demo(output_root:Path=Path('lesson_74_outputs'),seed:int=7201):
    torch.set_num_threads(1)
    require(type(seed) is int and seed>=0,'seed非法')
    folder=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    folder.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'python':platform.python_version(),'torch':str(torch.__version__),
              'device':'cpu','seed':seed,'status':'running','sources':SOURCES,
              'boundaries':['real_CPU_KV_Tensor_reuse_with_full_logits_equivalence',
                  'server_issued_secret_scopes_not_actual_HTTP_authentication',
                  'real_KV_pins_and_LRU_evictions_not_GPU_PagedAttention',
                  'single_process_no_thread_race_guarantees',
                  'media_bytes_only_key_metadata_not_real_multimodal_model',
                  'only_full_blocks_cacheable','no_wallclock_latency_or_GPU_benchmark',
                  'replay_rebuilds_empty_pool_not_live_session_checkpoint']}
    write_json(folder/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        save_models(folder/'models.pt',tok,models,seed)
        weights={'model_hashes':{v:state_hash(m.state_dict()) for v,m in models.items()},
                 'tokenizer_hash':digest(tok.contract()),'base_hash':base_hash(models['v0001']),
                 'model_seed':seed}
        write_json(folder/'weights.json',weights)
        report={'correctness.json':prefix_correctness_demo(seed),
                'invalidation.json':invalidation_demo(seed),
                'eviction.json':eviction_and_failure_demo(seed)}
        for name,payload in report.items():write_json(folder/name,payload)
        reload=reload_run(folder);write_json(folder/'reload.json',reload)
        manifest['status']='completed';write_json(folder/'manifest.json',manifest)
        print('完整演练输出:',folder)
        print('同作用域warm命中token:',report['correctness.json']['scenarios'][1]['hit_tokens'])
        print('跨租户命中token:',next(r['hit_tokens'] for r in report['correctness.json']['scenarios']
                                    if r['scenario']=='different_tenant_miss'))
        print('前缀/full logits最大差异:',report['correctness.json']['max_logit_error'])
        print('最后物理页占用:',report['correctness.json']['after_flush']['live_blocks'])
        return folder
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(folder/'manifest.json',manifest)
        raise

In [32]:
RUN_COMPLETE_DEMO = True
RUN_DIR = None
if RUN_COMPLETE_DEMO:
    RUN_DIR = fixture_demo(Path('lesson_74_outputs'),7201)
    write_json(RUN_DIR/'checks.json',checks)
else:
    print('完整演练已跳过；此处没有制造报告或模型结果。')

完整演练输出: lesson_74_outputs/run_20261104_133838_093131
同作用域warm命中token: 52
跨租户命中token: 0
前缀/full logits最大差异: 4.76837158203125e-07
最后物理页占用: 0


In [33]:
if RUN_DIR is not None:
    report=json.loads((RUN_DIR/'correctness.json').read_text(encoding='utf-8'))
    print('冷启动计算的token数:',report['scenarios'][0]['computed_tokens'])
    print('同范围重开命中的token数:',report['scenarios'][1]['hit_tokens'])
    print('跨租户命中数:',next(x['hit_tokens'] for x in report['scenarios']
                             if x['scenario']=='different_tenant_miss'))
    print('所有场景的最大logits误差:',report['max_logit_error'])
    print('最终活跃页:',report['after_flush']['live_blocks'])
else:
    print('未执行完整演练。')

冷启动计算的token数: 53
同范围重开命中的token数: 52
跨租户命中数: 0
所有场景的最大logits误差: 4.76837158203125e-07
最终活跃页: 0


## 20. 完全独立的只读重放

这个单元不复用上面模型对象，也没有从磁盘恢复正在使用中的KV页。它只从可信的本地权重重新构造不可变模型，核对分词与权重身份，然后在**全新空页池和新随机salt**下执行相同演练。

两次演练的salt不一样并不会影响应该命中/不命中的行为；也不会改模型数值。**不能把这些逻辑统计相等，当作真实跨机器缓存可直接复制的证明。**

In [34]:
if RUN_DIR is not None:
    print(json.dumps(reload_run(RUN_DIR),ensure_ascii=False,indent=2))

{
  "comparison": {
    "correctness.json": {
      "exact_report_equal": true
    },
    "invalidation.json": {
      "exact_report_equal": true
    },
    "eviction.json": {
      "exact_report_equal": true
    }
  },
  "weights_validated": true,
  "random_scopes_recreated_by_authority": true,
  "not_live_cache_session_resume": true
}


## 21. 课堂验收：能解释六个关键因果链

1. 为什么哈希中不只有块token，还需要**父块哈希与权重/模板/LoRA/隔离域**？
2. 如果两个prompt共享前27个token而block_size=4，为什么只能复用**24个token**？
3. 同一租户第二次命中28个token，为什么不意味着“实际TTFT快了28倍”？
4. 活动会话引用一页时，为什么撤销它的cache pin不能立刻清零该页？
5. 已缓存的第一个块没有变，换了模型版本号下的adapter权重，为什么仍必须miss？
6. 为什么服务器端随机salt+可信网关可以限制范围，而“让客户端传一个可猜tenant_id”不算授权？

**没有实现的部分：** 多线程/多进程原子性、真实鉴权网关、跨模型后端兼容校验、vLLM的GPU PagedAttention、真实图像编码器、KV量化/压缩、真实毫秒加速测量、跨节点KV传输与持久化、前缀逐出最优策略、访问时间侧信道的完整防御测试。

**下一课：第75课——推理缓存命中率、调度负载与容量压测。** 在本课的真实KV复用与租户隔离契约上，增加请求负载模型和可复现的容量/命中率报告，不用虚拟tick冒充GPU吞吐。

## 参考资料与方法边界

1. [vLLM Automatic Prefix Caching（块hash、父链、额外身份和完整块）](https://docs.vllm.ai/en/latest/design/prefix_caching/)
2. [vLLM Security（多租户前缀缓存、cache salt和隔离）](https://docs.vllm.ai/en/stable/usage/security/)
3. [Inside vLLM：模型调度、prefill与block命中](https://vllm.ai/blog/2025-09-05-anatomy-of-vllm)
4. [PagedAttention论文](https://arxiv.org/abs/2309.06180)
5. [Transformers Cache 解释（位置与逐层KV）](https://huggingface.co/docs/transformers/main/en/cache_explanation)
6. [PyTorch inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)
7. [PyTorch 可复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)

本课只运行真实的微型CPU模型与随机生成的示例请求，不使用外部真实用户隐私数据；哈希与模型相等性经过测试，但不构成实际生产环境的安全验证或性能承诺。